In [668]:
#| default_exp llm_core.call_llm

In [669]:
#| export
import re
from typing import Any, List, Literal, Optional, overload, TypedDict, Union, Any, Protocol, runtime_checkable

from openai import OpenAI
import lmstudio


import time
from typing import List, Optional, Any, Tuple

In [670]:
from fastcore.test import *
import time

from unittest.mock import patch

This module handles the general logic for calling llm's and processing their outputs.

## Separating thoughts from actual response for reasoning models

In [671]:
#| export
# --- Helper Parsers ---
#| export
def _parse_xml_tags(text: str) -> Optional[Tuple[str, str]]:
    """
    Handles <think> or <thought> tags.
    Resilient enough to catch truncated thoughts at the start of a string
    without misidentifying internal malformed tags as thoughts.
    """
    tag_pattern = r"<(think|thought)>([\s\S]*?)<\/\1>"
    
    # 1. Standard Case: Perfect pairs
    match = re.search(tag_pattern, text, re.IGNORECASE)
    if match:
        thoughts = match.group(2).strip()
        answer = re.sub(tag_pattern, "", text, flags=re.IGNORECASE).strip()
        return thoughts, answer

    # 2. Orphaned closing tag (The "Leak" Fix)
    close_pattern = r"<\/(think|thought)>"
    close_match = re.search(close_pattern, text, re.IGNORECASE)
    if close_match:
        split_idx = close_match.start()
        thoughts = text[:split_idx].strip()
        # Clean out any stray opening tag if it exists
        thoughts = re.sub(r"<(think|thought)>", "", thoughts, flags=re.IGNORECASE).strip()
        answer = text[close_match.end():].strip()
        return thoughts, answer

    # 3. Smart Truncation: Catch unclosed <think> ONLY if it starts the message
    # This satisfies ex_truncated_xml while allowing internal malformed tags to be ignored
    if text.strip().lower().startswith(("<think>", "<thought>")):
        opening_tag_len = text.find(">") + 1
        return text[opening_tag_len:].strip(), ""

    return None

In [672]:
#| export
def _parse_qwen_prose(text: str) -> Optional[Tuple[str, str]]:
    marker = "Thinking Process:"
    lowered_text = text.lower()
    start_idx = lowered_text.find(marker.lower())
    
    has_marker = start_idx != -1
    content = text[start_idx + len(marker):] if has_marker else text

    # We make the colon optional by adding :? 
    # This ensures it catches **Header**: and **Header**
    struct_patterns = [
        r"\s+#{1,3}\s",        
        r"\s+\*{2}.+\*{2}:?",   
        r"\s+-{3,}"            
    ]
    
    combined_regex = "|".join(struct_patterns)
    match = re.search(combined_regex, content)
    
    if match:
        split_idx = match.start()
        thoughts = content[:split_idx].strip()
        answer = content[split_idx:].strip()
        
        if not thoughts and not has_marker:
            return None
        return (thoughts or None), answer

    # Fallback to double-newline only if marker exists
    if has_marker:
        trimmed = content.strip()
        if "\n\n" in trimmed:
            parts = trimmed.rsplit("\n\n", 1)
            return parts[0].strip(), parts[1].strip()
        return trimmed, ""

    return None

In [673]:
#| export
def _parse_header_structure(text: str) -> Optional[Tuple[str, str]]:
    """Handles explicit THOUGHTS: / ANSWER: headers."""
    if "THOUGHTS:" in text.upper() and "ANSWER:" in text.upper():
        parts = re.split(r"THOUGHTS:", text, flags=re.IGNORECASE)
        # parts[1] is everything after "THOUGHTS:"
        sub_parts = re.split(r"ANSWER:", parts[1], flags=re.IGNORECASE)
        return sub_parts[0].strip(), sub_parts[1].strip()
    return None

In [674]:
#| export
import re

def _parse_channel_structure(raw_content: str) -> Optional[Tuple[str, str]]:
    """
    Parses <|channel|>analysis...<|channel|>final structure, 
    cleaning out injected 'assistant' labels and control tags.
    """
    # The marker usually looks like <|end|><|start|>assistant<|channel|>final<|message|>
    # We use a regex that identifies the transition to the final message.
    marker_pattern = r"<\|end\|>\s*<\|start\|>\s*(?:assistant)?\s*<\|channel\|>final<\|message\|>"
    
    # If the standard marker isn't there, check for just the final channel tag
    if not re.search(marker_pattern, raw_content):
        marker_pattern = r"<\|channel\|>final<\|message\|>"

    parts = re.split(marker_pattern, raw_content, maxsplit=1)
    
    if len(parts) < 2:
        # Check for truncated analysis
        if "<|channel|>analysis<|message|>" in raw_content:
            thoughts = raw_content.split("<|channel|>analysis<|message|>")[-1]
            return re.sub(r"<\|.*?\|>", "", thoughts).strip(), ""
        return None

    raw_thoughts, raw_answer = parts[0], parts[1]

    # Clean Thoughts: Remove the opening analysis tag and any leading <|start|>
    # We also remove the literal word 'assistant' if it leaked in at the start
    clean_thoughts = re.sub(r"<\|channel\|>analysis<\|message\|>", "", raw_thoughts)
    clean_thoughts = re.sub(r"<\|.*?\|>", "", clean_thoughts)
    clean_thoughts = re.sub(r"^assistant", "", clean_thoughts.strip()).strip()

    # Clean Answer: Just remove trailing control tags
    clean_answer = re.sub(r"<\|.*?\|>", "", raw_answer).strip()

    return clean_thoughts, clean_answer

In [675]:
#| export
import re
from typing import Optional, Tuple, Callable, List

def separate_thoughts(raw_content: str) -> Tuple[Optional[str], str]:
    """
    Orchestrates the separation of reasoning from content using 
    specialized helper parsers.
    """
    if not raw_content:
        return None, ""

    # List of parser functions to try in order of specificity
    parsers: List[Callable[[str], Optional[Tuple[str, str]]]] = [
        _parse_channel_structure,
        _parse_xml_tags,           # <think>...</think>
        _parse_header_structure,    # THOUGHTS: ... ANSWER: ...
        _parse_qwen_prose,         # Fuzzy/Prose logic (now acts as a catch-all)
    ]

    for parser in parsers:
        result = parser(raw_content)
        if result:
            return result

    # Final fallback: Everything is the answer
    return None, raw_content.strip()



The ```separate_thoughts``` function is used to extract the thought process and the final output of reasoning models (e.g. `DeepSeek-R1`, `Qwen3`, `GPT-5.2 Thinking`, etc.)

In [676]:
ex1 = """First, analyze the problem.<think>Step 1: Identify key components. Step 2: Validate inputs.</think> The final answer is 42."""

thoughts1, answer1 = separate_thoughts(ex1)
test_eq(thoughts1, "Step 1: Identify key components. Step 2: Validate inputs.")
test_eq(answer1, "First, analyze the problem. The final answer is 42.")

In [677]:
ex2 = """<THOUGHT>
Mathematical reasoning: check base cases first, then induction step.
Edge case x=0 gives y=1.
</THOUGHT>
Final computation: ∫[0,1] x² dx = 1/3"""

thoughts2, answer2 = separate_thoughts(ex2)
test_eq(thoughts2, "Mathematical reasoning: check base cases first, then induction step.\nEdge case x=0 gives y=1.")
test_eq(answer2, "Final computation: ∫[0,1] x² dx = 1/3")

In [678]:
ex3 = """THOUGHTS: Gal(L/K) is defined via étale cohomology. Verify separability first.
ANSWER: The Galois group Gal(L/K) is finite of order [L:K]."""

thoughts3, answer3 = separate_thoughts(ex3)
test_eq(thoughts3, "Gal(L/K) is defined via étale cohomology. Verify separability first.")
test_eq(answer3, "The Galois group Gal(L/K) is finite of order [L:K].")

In [679]:
# Example 4: Qwen 3.5 / LM Studio style (Thinking Process header)
ex4 = """Thinking Process:
1. Analyze the 'L-function' tag.
2. Identify ambient environment: Modular Forms.
3. Apply Adjective-Noun rule.

L-function associated with a modular form"""

thoughts4, answer4 = separate_thoughts(ex4)
test_eq(thoughts4, "1. Analyze the 'L-function' tag.\n2. Identify ambient environment: Modular Forms.\n3. Apply Adjective-Noun rule.")
test_eq(answer4, "L-function associated with a modular form")

# Example 5: Truncated Qwen Output (What happened in your first message)
ex5 = """Thinking Process:
1. Identify L-function as Dirichlet series.
2. Context is Analytic Number Theory."""

thoughts5, answer5 = separate_thoughts(ex5)
test_eq(thoughts5, "1. Identify L-function as Dirichlet series.\n2. Context is Analytic Number Theory.")
test_eq(answer5, "") # Returns empty because the model hit max_tokens before the answer

In [680]:
# Use raw strings (r"") for both input and expected output to preserve backslashes
ex6 = r"""<|channel|>analysis<|message|>We need to identify the notation.
The symbol $\nu$ is a positive integer. $Sel_\nu(A)$ is the Selmer group.
Thus output status.<|end|><|start|>assistant<|channel|>final<|message|>Status: No new contextual instantiations."""

thoughts6, answer6 = separate_thoughts(ex6)

# Match the exact line breaks of the source text
expected_thoughts = (
    "We need to identify the notation.\n"
    "The symbol $\\nu$ is a positive integer. $Sel_\\nu(A)$ is the Selmer group.\n"
    "Thus output status."
)

test_eq(thoughts6, expected_thoughts)
test_eq(answer6, "Status: No new contextual instantiations.")

In [681]:
#| hide
from fastcore.test import *

# Tag extraction works
test_eq(separate_thoughts("<think>reasoning</think>")[0], "reasoning")
test_eq(separate_thoughts("<THOUGHT>test</thought>")[0], "test")

# Answer extraction - EXACT original behavior
test_eq(separate_thoughts("A<think>B</think>C")[1], "AC")
test_eq(separate_thoughts("A <think>B</think> C")[1], "A  C")
test_eq(separate_thoughts("A\n<think>B</think>\nC")[1], "A\n\nC")

# Multiple tags - re.sub removes FIRST match + .strip()
multi = "<think>first</think>text<think>second</think>"
test_eq(separate_thoughts(multi)[0], "first")
test_eq(separate_thoughts(multi)[1], "text")  # ← Fixed: .strip() removes trailing tag

# No tags
test_eq(separate_thoughts("plain text")[0], None)
test_eq(separate_thoughts("plain text")[1], "plain text")

# Malformed tags
# Change line 23 to expect 'no close' to be captured as a thought
test_eq(separate_thoughts("<think>no close")[0], "no close")
test_eq(separate_thoughts("<think>no close")[1], "")

# THOUGHTS:ANSWER: fallback
test_eq(separate_thoughts("THOUGHTS: abc ANSWER: def"), ("abc", "def"))

# Empty cases
test_eq(separate_thoughts("<think></think>")[0], "")
test_eq(separate_thoughts("")[1], "")
test_eq(separate_thoughts("   ")[1], "")

In [682]:
#| hide
from fastcore.test import *

# --- Qwen 3.5 / LM Studio specific (Thinking Process:) ---

# Basic extraction
qwen_basic = "Thinking Process:\nAnalyze the term.\n\nFinal Noun Phrase"
test_eq(separate_thoughts(qwen_basic)[0], "Analyze the term.")
test_eq(separate_thoughts(qwen_basic)[1], "Final Noun Phrase")

# Handling multi-line thinking blocks
qwen_multiline = """Thinking Process:
1. Identify L-function as Dirichlet series.
2. Context is Analytic Number Theory.

L-function in analytic number theory"""
thoughts, answer = separate_thoughts(qwen_multiline)
test_eq(thoughts, "1. Identify L-function as Dirichlet series.\n2. Context is Analytic Number Theory.")
test_eq(answer, "L-function in analytic number theory")

# Truncated Thinking (Max Tokens hit before answer)
# This mimics the "PredictionResult" you saw earlier
qwen_truncated = """Thinking Process:
1. Analyze the 'L-function' tag.
2. Identify environment..."""
test_eq(separate_thoughts(qwen_truncated)[0], "1. Analyze the 'L-function' tag.\n2. Identify environment...")
test_eq(separate_thoughts(qwen_truncated)[1], "") # Answer is empty because no \n\n was reached

# Case Insensitivity (if your parser uses re.IGNORECASE or .lower())
qwen_case = "THINKING PROCESS: logic\n\nresult"
# Note: If your helper uses exact string matching "Thinking Process:", 
# you might need to adjust it to handle case-insensitivity.
test_eq(separate_thoughts(qwen_case)[0], "logic")
test_eq(separate_thoughts(qwen_case)[1], "result")

# --- Edge Cases with Mixed Logic ---

# Qwen marker inside an XML tag (XML should take priority)
mixed = "<think>Thinking Process: internal</think>Actual Answer"
test_eq(separate_thoughts(mixed)[0], "Thinking Process: internal")
test_eq(separate_thoughts(mixed)[1], "Actual Answer")

# Thinking Process marker with no double-newline content
# (Treats the whole block after marker as thought)
no_separator = "Thinking Process: Just some rambling thoughts without a clear answer."
test_eq(separate_thoughts(no_separator)[0], "Just some rambling thoughts without a clear answer.")
test_eq(separate_thoughts(no_separator)[1], "")

# --- Verification of Original Behavior Stability ---

# Ensure THOUGHTS: fallback still works even with new Qwen logic present
legacy_fallback = "THOUGHTS: step 1 ANSWER: step 2"
test_eq(separate_thoughts(legacy_fallback), ("step 1", "step 2"))

In [683]:
# Answer comes before the thinking block
answer_first = "The final answer is 42. <think>I calculated this by multiplying 6 and 7.</think>"
test_eq(separate_thoughts(answer_first)[1], "The final answer is 42.")
test_eq(separate_thoughts(answer_first)[0], "I calculated this by multiplying 6 and 7.")

In [684]:
# Qwen with complex thinking structure
qwen_complex = """Thinking Process:
Step 1: Analyze.

Step 2: Synthesize.

Final Result"""
thoughts, answer = separate_thoughts(qwen_complex)
test_eq(thoughts, "Step 1: Analyze.\n\nStep 2: Synthesize.")
test_eq(answer, "Final Result")

In [685]:
# Marker words used in plain prose
plain_prose = "I am currently in the thinking process of writing my answer."
test_eq(separate_thoughts(plain_prose)[0], None)
test_eq(separate_thoughts(plain_prose)[1], "I am currently in the thinking process of writing my answer.")

In [686]:
# ex_qwen_standard
qwen_input = r"""Thinking Process:
The user wants an analysis of the sheaf $\mathscr{F}$.
I should check if it's symplectically self-dual.

The definition is consistent with the previous context.

Primary Objects: $\mathscr{F}$"""

thoughts, answer = separate_thoughts(qwen_input)
test_eq(thoughts, "The user wants an analysis of the sheaf $\\mathscr{F}$.\nI should check if it's symplectically self-dual.\n\nThe definition is consistent with the previous context.")
test_eq(answer, r"Primary Objects: $\mathscr{F}$")

In [687]:
# ex_qwen_truncated
qwen_truncated = """Thinking Process:
1. Start extracting objects.
2. Found $k$ as a field.
3. Found $C$ as a curve.
4. Analyzing the lemma's hypotheses regarding $\nu$."""

thoughts, answer = separate_thoughts(qwen_truncated)
test_eq(thoughts, "1. Start extracting objects.\n2. Found $k$ as a field.\n3. Found $C$ as a curve.\n4. Analyzing the lemma's hypotheses regarding $\nu$.")
test_eq(answer, "")

In [688]:
# ex_mixed_style
mixed_input = """<think>
Checking for $H^1$ vanishing.
</think>
Thinking Process:
Wait, I already used tags above.

The final result is null."""

thoughts, answer = separate_thoughts(mixed_input)
# Since _parse_xml_tags is usually higher in the list, it should win.
test_eq(thoughts, "Checking for $H^1$ vanishing.")
test_eq(answer, "Thinking Process:\nWait, I already used tags above.\n\nThe final result is null.")

In [689]:
# ex_math_preservation
math_input = r"""Thinking Process:
Check if $\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}$.

$\text{curl}(\mathbf{E}) = -\mathbf{\dot{B}}$"""

thoughts, answer = separate_thoughts(math_input)
test_eq(thoughts, r"Check if $\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}$.")
test_eq(answer, r"$\text{curl}(\mathbf{E}) = -\mathbf{\dot{B}}$")

In [690]:
# ex_legacy_explicit
legacy_input = """THOUGHTS: 
* Item A
* Item B

ANSWER:
Final result here."""

thoughts, answer = separate_thoughts(legacy_input)
test_eq(thoughts, "* Item A\n* Item B")
test_eq(answer, "Final result here.")

In [691]:
# --- Structural Splitting Tests ---

# ex_bold: Note that thoughts are stripped of trailing newlines
ex_bold = """Thinking Process:
I need to identify the new objects in this passage.
I will check for sheaves and points.

**New Contextual Updates:**
* Primary Objects: $b$"""

thoughts, answer = separate_thoughts(ex_bold)
test_eq(thoughts, "I need to identify the new objects in this passage.\nI will check for sheaves and points.")
test_eq(answer, "**New Contextual Updates:**\n* Primary Objects: $b$")

# ex_head: Splits at the ### header
ex_head = """Thinking Process:
Analyze the Galois group structure.

### Final Result
The group is cyclic."""

thoughts, answer = separate_thoughts(ex_head)
test_eq(thoughts, "Analyze the Galois group structure.")
test_eq(answer, "### Final Result\nThe group is cyclic.")

# --- Fallback/Priority Tests ---

# ex4: This uses the rsplit("\n\n") fallback because no # or ** header exists
ex4_fixed = """Thinking Process:
1. Analyze the 'L-function' tag.
2. Identify ambient environment: Modular Forms.
3. Apply Adjective-Noun rule.

L-function associated with a modular form"""

t4, a4 = separate_thoughts(ex4_fixed)
test_eq(t4, "1. Analyze the 'L-function' tag.\n2. Identify ambient environment: Modular Forms.\n3. Apply Adjective-Noun rule.")
test_eq(a4, "L-function associated with a modular form")

# ex_legacy_priority: This should be handled by _parse_header_structure BEFORE qwen_prose
ex_legacy = """THOUGHTS: 
* Item A
* Item B

ANSWER:
Final result here."""

tl, al = separate_thoughts(ex_legacy)
test_eq(tl, "* Item A\n* Item B")
test_eq(al, "Final result here.")

# --- Edge Cases & Robustness ---

# ex_hr: Splits at the horizontal rule
ex_hr = """Thinking Process:
Finalizing the derivation.
---
$y = mx + b$"""

thoughts, answer = separate_thoughts(ex_hr)
test_eq(thoughts, "Finalizing the derivation.")
test_eq(answer, "---\n$y = mx + b$")

# ex_no_mark: Testing the "Catch-all" behavior for structural headers
ex_no_mark = """I am thinking about the proof.

**Primary Objects**: $A$"""

thoughts, answer = separate_thoughts(ex_no_mark)
# With the catch-all logic, it should split even without the "Thinking Process" label
test_eq(thoughts, "I am thinking about the proof.")
test_eq(answer, "**Primary Objects**: $A$")

# ex_multi_para: Testing that internal paragraphs in the answer don't cause a split
ex_multi_para = """Thinking Process:
Check the fiber.

**New Contextual Updates:**
* Obj 1

* Obj 2"""

thoughts, answer = separate_thoughts(ex_multi_para)
test_eq(thoughts, "Check the fiber.")
# The answer starts at the first structural header and takes everything else
test_eq(answer, "**New Contextual Updates:**\n* Obj 1\n\n* Obj 2")

In [692]:
#| hide
from fastcore.test import *

# --- 1. XML Resilience Tests (The "Anti-Leak" Suite) ---

# Orphaned Closing Tag: Catching the logic even if the model misses the opener
ex_orphan = "This was reasoning but I missed the start tag. </think> Actual Answer starts here."
thoughts, answer = separate_thoughts(ex_orphan)
test_eq(thoughts, "This was reasoning but I missed the start tag.")
test_eq(answer, "Actual Answer starts here.")

# Truncated Thinking: Model cut off before finishing the thought
ex_truncated_xml = "<think> I am currently deriving the property of the sheaf"
thoughts, answer = separate_thoughts(ex_truncated_xml)
test_eq(thoughts, "I am currently deriving the property of the sheaf")
test_eq(answer, "")

# Nested/Malformed: Ensuring multiple tags don't leak into the answer
ex_nested = "<think>Logic A</think> <think>Logic B</think> Final Result"
thoughts, answer = separate_thoughts(ex_nested)
test_eq(thoughts, "Logic A") # Should take first match
test_eq(answer, "Final Result") # Should clean ALL tags out

# --- 2. Qwen/LM Studio Prose Tests (The "Silver Platter" Suite) ---

# Qwen Standard: Testing the split between thinking and bold headers
# Use r""" for the multi-line string
ex_qwen = r"""Thinking Process:
Identify the valuation ring.

**New Contextual Updates:**
* Object: $\mathcal{O}_K$"""

thoughts, answer = separate_thoughts(ex_qwen)
test_eq(thoughts, "Identify the valuation ring.")

# Use r"" for the expected answer string
test_eq(answer, r"**New Contextual Updates:**" + "\n" + r"* Object: $\mathcal{O}_K$")
# OR more simply:
test_eq(answer, r"**New Contextual Updates:**" + "\n" + r"* Object: $\mathcal{O}_K$")

# Qwen Truncated: No answer yet
ex_qwen_trunc = """Thinking Process:
Check for etale descent..."""
thoughts, answer = separate_thoughts(ex_qwen_trunc)
test_eq(thoughts, "Check for etale descent...")
test_eq(answer, "")

# --- 3. Legacy & Fallback Stability ---

# Mixed Styles: XML should take priority over Prose Marker
ex_mixed = "<think>Step 1</think> Thinking Process: Step 2\n\nFinal Answer"
thoughts, answer = separate_thoughts(ex_mixed)
test_eq(thoughts, "Step 1") # XML is higher in the parser list
test_eq(answer, "Thinking Process: Step 2\n\nFinal Answer")

# Plain Prose: No reasoning at all
test_eq(separate_thoughts("Just a normal math definition.")[0], None)
test_eq(separate_thoughts("Just a normal math definition.")[1], "Just a normal math definition.")

# --- 4. The "Channel" Complex Structure ---

ex_channel = r"""<|channel|>analysis<|message|>We need to identify the notation.
The symbol $\nu$ is a positive integer.<|end|><|start|>assistant<|channel|>final<|message|>Status: Done."""
thoughts, answer = separate_thoughts(ex_channel)
test_eq(thoughts, "We need to identify the notation.\nThe symbol $\\nu$ is a positive integer.")
test_eq(answer, "Status: Done.")

## Handling different model API/libraries

In [693]:
#| export

@runtime_checkable
class LLMProvider(Protocol):
    """Structural requirement for a model to be used in this script."""
    def respond(self, payload: dict, config: Optional[dict] = None) -> Any: ...
    @property
    def chat(self) -> Any: ...

# The Type Alias: This is what the user sees in their IDE hover-text.
# It explicitly lists the intended classes + our generic Protocol.
SupportedLLM = Union[lmstudio.LLM, OpenAI, LLMProvider]

In [694]:
#| export
def smart_truncate(
    text: str, 
    model: SupportedLLM,  # Using the alias here
    max_context: int, 
    reserved_tokens: int, 
    verbose: bool = False
) -> str:
    r"""Truncates `text` based on a herustic (3 chars per token) to prevent
    context window overflow."""
    # Note: 'hasattr' still works great here for logic branching
    if hasattr(model, 'tokenize'):
        available_tokens = max_context - reserved_tokens
        char_limit = available_tokens * 3 
    else:
        available_tokens = max_context - reserved_tokens
        char_limit = available_tokens * 3
        
    if len(text) > char_limit:
        if verbose: print(f"Warning: Text truncated to ~{char_limit} chars.")
        return text[:char_limit] + "..."
    return text


In [695]:
# Example
class MockModel: tokenize = True
text = "This is a very long string of text."
truncated = smart_truncate(text, MockModel(), max_context=10, reserved_tokens=5)
print(truncated)

This is a very ...


In [696]:
#| hide
from fastcore.test import *

# Test basic truncation
test_eq(smart_truncate("Hello World", None, 10, 8), "Hello ...") # 2 tokens * 3 = 6 chars
# Test no truncation needed
test_eq(smart_truncate("Short", None, 100, 10), "Short")

## Calling the LLM and handling its response

In [697]:
#| export
def _parse_groq_limits(headers: Any) -> dict:
    """Extracts Groq-specific rate limit info from response headers."""
    return {
        "remaining_requests": headers.get("x-ratelimit-remaining-requests"),
        "remaining_tokens": headers.get("x-ratelimit-remaining-tokens"),
        "reset_requests": headers.get("x-ratelimit-reset-requests"),
        "reset_tokens": headers.get("x-ratelimit-reset-tokens"),
        "retry_after": headers.get("retry-after")
    }

In [698]:
#| export
def _extract_headers(headers: Any) -> dict:
    """Safely extracts rate limits; returns empty dict if not present."""
    if not headers: return {}
    # Use .get() to avoid KeyErrors if these aren't Groq headers
    return {
        "rem_req": headers.get("x-ratelimit-remaining-requests"),
        "rem_tok": headers.get("x-ratelimit-remaining-tokens"),
        "reset": headers.get("x-ratelimit-reset-requests")
    }

In [699]:
#| hide
# Test Header Extraction
standard_headers = {"Content-Type": "application/json"}
groq_headers = {
    "x-ratelimit-remaining-tokens": "5000",
    "x-ratelimit-reset-requests": "10s"
}

# Standard headers should return empty values for limits, not crash
test_eq(_extract_headers(standard_headers).get('rem_tok'), None)

# Groq headers should map correctly
extracted = _extract_headers(groq_headers)
test_eq(extracted['rem_tok'], "5000")
test_eq(extracted['reset'], "10s")

# Null case
test_eq(_extract_headers(None), {})

In [700]:
#| export
def _get_input_metrics(messages: List[dict]) -> Tuple[float, str, int]:
    """Returns (perf_counter, timestamp_string, character_count)."""
    return time.perf_counter(), time.strftime("%H:%M:%S"), sum(len(m['content']) for m in messages)

In [701]:
#| export

def _handle_openai_call(model: Any, messages: List[dict], config: dict) -> Tuple[str, Optional[float], Any]:
    """Handles OpenAI streaming with fallback and safe limit extraction."""
    full_content, ttft, start_perf, usage = "", None, time.perf_counter(), None
    try:
        # We use with_raw_response to wrap the stream so we can peek at headers
        raw = model.chat.completions.with_raw_response.create(
            messages=messages, stream=True, stream_options={"include_usage": True}, **config
        )
        limits = _extract_headers(raw.headers) # Capture headers immediately
        for chunk in raw.parse():
            if not ttft and chunk.choices and chunk.choices[0].delta.content:
                ttft = time.perf_counter() - start_perf
            if chunk.choices and chunk.choices[0].delta.content:
                full_content += chunk.choices[0].delta.content
            if chunk.usage: usage = chunk.usage
        if usage: usage.limits = limits # Attach limits to usage
        return full_content, ttft, usage
    except Exception:
        raw_fallback = model.chat.completions.with_raw_response.create(
            messages=messages, stream=False, **config
        )
        res = raw_fallback.parse()
        res.usage.limits = _extract_headers(raw_fallback.headers)
        return res.choices[0].message.content, None, res.usage

In [702]:
#| export
def _handle_lms_call(model: Any, messages: List[dict], config: dict) -> Tuple[str, Optional[float], Any]:
    """Handles LM Studio streaming with a safe fallback."""
    full_content, ttft, start_perf = "", None, time.perf_counter()
    lms_config = {**config, "maxTokens": config.get("max_tokens", 8192)}
    try:
        stream = model.respond({"messages": messages}, config=lms_config, stream=True)
        for chunk in stream:
            if ttft is None: ttft = time.perf_counter() - start_perf
            full_content += getattr(chunk, 'content', "")
        return full_content, ttft, getattr(stream, 'usage', {})
    # except Exception:
    except Exception as e:
        # if verbose:
        #     print(f"[Debug] Streaming failed because: {e}")
        # ... rest of fallback code ...
        res = model.respond({"messages": messages}, config=lms_config)
        return getattr(res, 'content', str(res)), None, getattr(res, 'usage', {})


In [703]:
#| export
def _log_llm_stats(
        out: str,
        in_c: int,
        dur: float,
        ttft: float,
        usage: Any,
        start_t: str):
    """Prints performance and usage metrics."""
    u = usage if isinstance(usage, dict) else getattr(usage, '__dict__', {})
    comp_tokens = u.get('completion_tokens', 0) or getattr(usage, 'completion_tokens', 0)
    tps = comp_tokens / dur if dur > 0 else 0
    
    print(f"\n[LLM] In: {in_c}c | Out: {len(out)}c | Time: {start_t} -> {time.strftime('%H:%M:%S')}")
    print(f"[LLM] Dur: {dur:.2f}s | TTFT: {f'{ttft:.2f}s' if ttft else 'N/A'} | TPS: {tps:.2f}")
    
    if hasattr(usage, 'limits') and usage.limits.get('rem_tok'):
        print(f"[Limits] Remaining Tokens: {usage.limits['rem_tok']} | Reset: {usage.limits['reset']}")

In [704]:
#| export
def call_llm(
    model: 'SupportedLLM',
    messages: List[dict],
    config: Optional[dict] = None,
    verbose: bool = False,
    return_usage: bool = False
) -> str | Tuple[str, Any]:
    """Calls the LLM and optionally returns usage metadata."""
    conf = {"temperature": 0.1, "max_tokens": 8192, **(config or {})}
    start_p, start_t, in_chars = _get_input_metrics(messages)
    
    if hasattr(model, 'respond'):
        out, ttft, usage = _handle_lms_call(model, messages, conf)
    elif hasattr(model, 'chat'):
        m_name = conf.pop("model_name", "gpt-4o")
        out, ttft, usage = _handle_openai_call(model, messages, {"model": m_name, **conf})
    else:
        raise ValueError("Unsupported model interface.")

    dur = time.perf_counter() - start_p
    if verbose:
        _log_llm_stats(out, in_chars, dur, ttft, usage, start_t)
    
    return (out.strip(), usage) if return_usage else out.strip()

In [705]:
# Example Mocking LM Studio
class MockLMS: 
    def respond(self, p, config=None): return "Direct Response String"

messages = [{"role": "user", "content": "Hi"}]
print(call_llm(MockLMS(), messages))

Direct Response String


In [706]:
#| hide
# 1. Fix the Mock names and make them robust
class MockLMS:
    """Simulates a model that returns a simple string or object."""
    def respond(self, payload, config=None, stream=False):
        # Your helper expects an iterable if stream=True
        if stream:
            class Chunk: content = "Object Response"
            return [Chunk()] 
        
        class Res: 
            content = "Object Response"
            usage = {"completion_tokens": 10}
        return Res()

# 2. Update the tests
# Ensure the mock name matches: MockLMS()
test_eq(call_llm(MockLMS(), [{"role":"user", "content":"hi"}]), "Object Response")

# 3. Ensure the failure test provides the expected input structure
test_fail(lambda: call_llm("NotAModel", [{"role":"user", "content":"hi"}]), contains="Unsupported model interface")

In [707]:
#| hide
class MockStreamingModel:
    def respond(self, payload, config=None, stream=False):
        class Chunk:
            def __init__(self, content): self.content = content
        
        # This class acts as the 'Prediction' object
        class Prediction:
            def __init__(self):
                self.usage = {"completion_tokens": 2}
            def __iter__(self):
                # This makes the object 'Loop-able'
                time.sleep(0.1) # TTFT delay
                yield Chunk("Streaming ")
                yield Chunk("Response")

        if stream:
            return Prediction()
        
        # Non-streaming fallback
        class Res:
            content = "Static Response"
            usage = {"completion_tokens": 2}
        return Res()

In [708]:
#| hide
class MockUsage:
    def __init__(self):
        self.completion_tokens = 5
        self.limits = {"rem_tok": "100"}

class MockModelWithUsage:
    """Mocks an OpenAI-like model that provides usage data."""
    def respond(self, payload, config=None, stream=False):
        class Res:
            content = "Usage Test"
            usage = MockUsage()
        return Res()

messages = [{"role": "user", "content": "test"}]
model = MockModelWithUsage()

# Test 1: Default behavior (returns str)
res_str = call_llm(model, messages, return_usage=False)
test_is(type(res_str), str)
test_eq(res_str, "Usage Test")

# Test 2: Return Usage behavior (returns Tuple)
res_tuple = call_llm(model, messages, return_usage=True)
test_is(type(res_tuple), tuple)
test_eq(len(res_tuple), 2)
test_eq(res_tuple[0], "Usage Test")
test_eq(res_tuple[1].completion_tokens, 5)

In [709]:
#| hide
class MockRawResponse:
    """Simulates the object returned by .with_raw_response.create()"""
    def __init__(self):
        self.headers = {"x-ratelimit-remaining-tokens": "999"}
    def parse(self):
        class Choice:
            message = type('obj', (object,), {'content': "Fallback result"})
        class Usage:
            completion_tokens = 10
        return type('obj', (object,), {'choices': [Choice()], 'usage': Usage()})

# Test that limits are attached in the fallback/non-streaming path
@patch('__main__._extract_headers')
def test_metadata_attachment(mock_ext):
    mock_ext.return_value = {"rem_tok": "999"}
    # This simulates the internal behavior of _handle_openai_call's except block
    raw_fallback = MockRawResponse()
    res = raw_fallback.parse()
    res.usage.limits = _extract_headers(raw_fallback.headers)
    
    test_eq(res.usage.limits['rem_tok'], "999")
    test_eq(res.usage.completion_tokens, 10)

test_metadata_attachment()

In [710]:
#| hide
# Test logger with dictionary (LM Studio style)
dict_usage = {"completion_tokens": 10, "limits": {"rem_tok": "50", "reset": "1s"}}
# Should not raise error
_log_llm_stats("out", 10, 1.0, 0.1, dict_usage, "12:00:00")

# Test logger with object (OpenAI style)
class ObjUsage:
    completion_tokens = 20
    limits = {"rem_tok": "100", "reset": "2s"}
# Should not raise error
_log_llm_stats("out", 10, 1.0, 0.1, ObjUsage(), "12:00:00")


[LLM] In: 10c | Out: 3c | Time: 12:00:00 -> 22:26:30
[LLM] Dur: 1.00s | TTFT: 0.10s | TPS: 10.00

[LLM] In: 10c | Out: 3c | Time: 12:00:00 -> 22:26:30
[LLM] Dur: 1.00s | TTFT: 0.10s | TPS: 20.00
[Limits] Remaining Tokens: 100 | Reset: 2s


In [711]:
#| export
class LLMResponse(TypedDict):
    r"""
    A `TypedDict` representing a processed response with separate logic and final output.

    Output of `process_llm_response` when `return_thoughts=True`
    """
    thoughts: str
    output: str

In [712]:
#| export
# Overload 1: If return_thoughts is True, return a dict
@overload
def process_llm_response(raw_text: str, return_thoughts: Literal[True]) -> LLMResponse: ...

# Overload 2: If return_thoughts is False (default), return a str
@overload
def process_llm_response(raw_text: str, return_thoughts: Literal[False] = False) -> str: ...

# The actual implementation (type hints here are usually more generic)
def process_llm_response(
    raw_text: str,
    return_thoughts: bool = False
) -> str | LLMResponse:
    """Separates thoughts and returns either a string or an `LLMResponse` dict."""
    if not raw_text: 
        return {"thoughts": "", "output": ""} if return_thoughts else ""
        
    thoughts, clean_answer = separate_thoughts(raw_text)
    
    if return_thoughts:
        # Cast to LLMResponse or just return; TypedDict validates the keys
        return {"thoughts": thoughts or "", "output": clean_answer}
    
    return clean_answer

In [713]:
# Example
raw = "<think>Calculating 2+2</think>The answer is 4."
print(f"String only: {process_llm_response(raw)}")
print(f"With thoughts: {process_llm_response(raw, return_thoughts=True)}")

String only: The answer is 4.
With thoughts: {'thoughts': 'Calculating 2+2', 'output': 'The answer is 4.'}


In [714]:
#| hide
# Note: separate_thoughts must be defined in the namespace for these to pass
raw_input = "<think>logic</think>answer"

# Test string return (default)
test_eq(process_llm_response(raw_input), "answer")

# Test Dict return
processed = process_llm_response(raw_input, return_thoughts=True)
test_is(type(processed), dict)
test_eq(processed['thoughts'], "logic")
test_eq(processed['output'], "answer")

# Test empty thoughts fallback
test_eq(process_llm_response("just answer", True)['thoughts'], "")